# Week 2 Concepts — LLM Internals, Audio AI, Sandboxing

Runnable companion notebook for Days 1-4. Each section uses a different
worked example than its matching daily note, demonstrating the same
underlying technique.

## Day 1: Tokenization, Attention, and the KV Cache

Below: comparing token counts for the same sentence in English and
Japanese, then a minimal from-scratch single-head self-attention pass
over a short toy sequence to make Query/Key/Value concrete.

In [ ]:
import tiktoken

enc = tiktoken.get_encoding("cl100k_base")
sentence_en = "Please confirm the meeting time before Friday."
sentence_ja = "金曜日までに会議の時間を確認してください。"

print(len(enc.encode(sentence_en)), "tokens for English")
print(len(enc.encode(sentence_ja)), "tokens for Japanese")

In [ ]:
import numpy as np

def softmax(x, axis=-1):
    x = x - np.max(x, axis=axis, keepdims=True)
    e = np.exp(x)
    return e / e.sum(axis=axis, keepdims=True)

# toy sequence of 4 tokens, embedding dim 8
np.random.seed(0)
seq_len, d_model = 4, 8
token_embeddings = np.random.randn(seq_len, d_model)

W_q = np.random.randn(d_model, d_model) * 0.1
W_k = np.random.randn(d_model, d_model) * 0.1
W_v = np.random.randn(d_model, d_model) * 0.1

Q = token_embeddings @ W_q
K = token_embeddings @ W_k
V = token_embeddings @ W_v

attention_weights = softmax((Q @ K.T) / np.sqrt(d_model))
context = attention_weights @ V   # new representation per token

attention_weights.round(2)

## Day 2: Embedding Models and Reranking

Below: a bi-encoder search over short product-support snippets where an
irrelevant-but-lexically-similar snippet outranks the relevant one, and
a cross-encoder rerank that fixes the ordering.

In [ ]:
from sentence_transformers import SentenceTransformer, CrossEncoder
import numpy as np

bi_encoder = SentenceTransformer("sentence-transformers/all-MiniLM-L6-v2")

query = "battery drains overnight even when the device is off"

snippets = [
    "If the battery drains while powered off, check for a stuck background update and disable auto-sync overnight.",  # relevant
    "Our extended battery pack ships separately and takes 4 hours to fully charge overnight.",                          # irrelevant, shares 'battery'/'overnight'
]

q_vec = bi_encoder.encode(query)
d_vecs = bi_encoder.encode(snippets)
bi_scores = d_vecs @ q_vec / (np.linalg.norm(d_vecs, axis=1) * np.linalg.norm(q_vec))
list(zip(snippets, bi_scores.round(3)))

In [ ]:
cross_encoder = CrossEncoder("cross-encoder/ms-marco-MiniLM-L-6-v2")
pairs = [(query, s) for s in snippets]
cross_scores = cross_encoder.predict(pairs)
list(zip(snippets, cross_scores.round(3)))

## Day 3: Audio AI — ASR/TTS

Below: transcribing a voicemail clip with Whisper, using an
`initial_prompt` hint to reduce a jargon misrecognition, then an
OS-level TTS fallback for reading a short status message aloud.

In [ ]:
import whisper

model = whisper.load_model("base")

# Without a hint, a product name like "Postgres" can come back mangled
result_plain = model.transcribe("voicemail_042.wav")
print("plain:", result_plain["text"])

# With a domain hint, decoding is biased toward the expected vocabulary
result_hinted = model.transcribe(
    "voicemail_042.wav",
    initial_prompt="Postgres, replica, failover, connection pool",
)
print("hinted:", result_hinted["text"])

In [ ]:
import pyttsx3

engine = pyttsx3.init()
engine.say("The nightly backup completed with no errors.")
engine.runAndWait()

## Day 4: Code-Execution Sandboxing for Agents

Below: the generic create/run/files/kill lifecycle a hosted sandbox SDK
exposes (schematic — provider method names vary), followed by a local
`subprocess` + `resource` limit pattern explicitly marked as a
speed bump, not a security boundary.

In [ ]:
# Schematic example of a hosted sandbox SDK's typical lifecycle.
# Illustrative only -- exact method names differ across providers.

sandbox = Sandbox.create(timeout=60, network_access=False)

output = sandbox.run_code("import statistics; print(statistics.mean([3, 7, 9, 12]))")

sandbox.files.write("/tmp/report.csv", csv_bytes)
sandbox.commands.run("pip install pandas")

sandbox.kill()

In [ ]:
import resource
import subprocess

def limit_resources():
    resource.setrlimit(resource.RLIMIT_CPU, (5, 5))                   # 5 CPU-seconds
    resource.setrlimit(resource.RLIMIT_AS, (256 * 1024 * 1024,) * 2)  # 256MB address space

generated_code = "print(sum(range(1000)))"

# NOTE: this bounds CPU time / memory only. It is not a real security
# boundary -- no filesystem or network isolation is provided here.
result = subprocess.run(
    ["python3", "-c", generated_code],
    timeout=10,
    preexec_fn=limit_resources,
    capture_output=True,
    text=True,
)
result.stdout, result.returncode